# Titanic Survival Prediction

Predicting passenger survival using decision trees and random forests, 
and figuring out which features actually drive the predictions versus 
which just look important.

## About the Data

Kaggle's Titanic dataset (`train.csv`) — 891 rows, 12 columns.

In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('train.csv')

In [3]:
df.shape

(891, 12)

In [4]:
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [5]:
df.isnull().sum().sort_values(ascending=False)

Cabin          687
Age            177
Embarked         2
PassengerId      0
Name             0
Pclass           0
Survived         0
Sex              0
Parch            0
SibSp            0
Fare             0
Ticket           0
dtype: int64

## Data Cleaning & Feature Engineering

Three columns had missing values: `Cabin` (687), `Age` (177), 
`Embarked` (2). Handled each based on why the data was likely missing.

### Embarked — fill and encode

In [6]:
df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)

C:\Users\nifem\AppData\Local\Temp\ipykernel_24600\3744086084.py:1: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Embarked'].fillna(df['Embarked'].mode()[0], inplace=True)


0      S
1      C
2      S
3      S
4      S
      ..
886    S
887    S
888    S
889    C
890    Q
Name: Embarked, Length: 891, dtype: str

In [7]:
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

### Sex — encode


In [ ]:
df['sex_encoded'] = df['Sex'].map({'male': 0, 'female': 1})

### Age — fill using median within each passenger class

Age varies a lot by class, so a single overall median would have 
been less accurate than filling per-`Pclass`.

In [9]:
df['Age'].isnull().sum()

np.int64(177)

In [10]:
df['Age'] = df.groupby('Pclass')['Age'].transform(lambda x: x.fillna(x.median()))

### Title — extracted from Name

Titles like `Mr`, `Mrs`, `Miss`, and `Master` carry information 
`Sex` and `Age` don't fully capture on their own (e.g. distinguishing 
young boys from adult men). Rare titles were grouped into a single 
`Rare` category to avoid creating columns with only 1-2 passengers.

In [11]:
df['Title'] = df['Name'].str.extract(r',\s*([^\.]*)\.')
df['Title'].value_counts()

Title
Mr              517
Miss            182
Mrs             125
Master           40
Dr                7
Rev               6
Major             2
Mlle              2
Col               2
Don               1
Mme               1
Ms                1
Lady              1
Sir               1
Capt              1
the Countess      1
Jonkheer          1
Name: count, dtype: int64

In [12]:
Rare_titles = ['Lady', 'Countess','Capt', 'Col','Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Mlle', 'Ms', 'Mme', 'the Countess']
df['Title'] = df['Title'].replace(Rare_titles, 'Rare')

In [13]:
df['Title'].value_counts()

Title
Mr        517
Miss      182
Mrs       125
Master     40
Rare       27
Name: count, dtype: int64

In [14]:
df = pd.get_dummies(df,columns = ['Title'])

### Cabin — converted to a Has_Cabin flag

687 of 891 values were missing, too many to use directly. Whether 
a cabin was recorded at all still carries signal, likely because 
cabin numbers were mainly logged for wealthier passengers.

In [15]:
df['Has_Cabin'] = df['Cabin'].notna().astype(int)
df.groupby('Has_Cabin')['Survived'].mean()

Has_Cabin
0    0.299854
1    0.666667
Name: Survived, dtype: float64

In [16]:
df.groupby('Pclass')['Has_Cabin'].mean()

Pclass
1    0.814815
2    0.086957
3    0.024440
Name: Has_Cabin, dtype: float64

### Final feature check

In [25]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked_Q', 'Embarked_S',
       'sex_encoded', 'Title_Master', 'Title_Miss', 'Title_Mr', 'Title_Mrs',
       'Title_Rare', 'Has_Cabin'],
      dtype='str')

In [26]:
df.shape


(891, 20)

## Model

Final features: `Pclass`, `sex_encoded`, `Age`, `Fare`, `Has_Cabin`, 
`Title_Miss`, `Title_Mr`, `Title_Mrs`, `Title_Rare`.

Compared a single Decision Tree (tuned via `max_depth`) against a 
Random Forest (100 trees).

In [19]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

features = ['Pclass', 'sex_encoded', 'Age', 'Fare','Has_Cabin', 'Title_Miss', 'Title_Mr', 'Title_Mrs','Title_Rare']

X = df[features]
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

for depth in range(1, 15):
    tree  =  DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, y_train)
    train_accuracy = accuracy_score(y_train, tree.predict(X_train))
    test_accuracy = accuracy_score(y_test, tree.predict(X_test))
    print(f"Max Depth: {depth}, Train Accuracy: {train_accuracy}, Test Accuracy: {test_accuracy}")

Max Depth: 1, Train Accuracy: 0.7823033707865169, Test Accuracy: 0.7821229050279329
Max Depth: 2, Train Accuracy: 0.7963483146067416, Test Accuracy: 0.7597765363128491
Max Depth: 3, Train Accuracy: 0.8286516853932584, Test Accuracy: 0.8156424581005587
Max Depth: 4, Train Accuracy: 0.8370786516853933, Test Accuracy: 0.8212290502793296
Max Depth: 5, Train Accuracy: 0.8469101123595506, Test Accuracy: 0.8379888268156425
Max Depth: 6, Train Accuracy: 0.8679775280898876, Test Accuracy: 0.7541899441340782
Max Depth: 7, Train Accuracy: 0.8834269662921348, Test Accuracy: 0.8379888268156425
Max Depth: 8, Train Accuracy: 0.9030898876404494, Test Accuracy: 0.770949720670391
Max Depth: 9, Train Accuracy: 0.9213483146067416, Test Accuracy: 0.8379888268156425
Max Depth: 10, Train Accuracy: 0.9382022471910112, Test Accuracy: 0.8212290502793296
Max Depth: 11, Train Accuracy: 0.952247191011236, Test Accuracy: 0.8268156424581006
Max Depth: 12, Train Accuracy: 0.9648876404494382, Test Accuracy: 0.83798882

In [ ]:
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
forest.fit(X_train, y_train)

print("Forest Train:", accuracy_score(y_train, forest.predict(X_train)))
print("Forest Test:", accuracy_score(y_test, forest.predict(X_test)))

Forest Train: 0.9396067415730337
Forest Test: 0.8435754189944135


## Findings & Results

In [21]:

important_features = pd.Series(forest.feature_importances_, index=features).sort_values(ascending=False)
print(important_features)


Fare           0.255350
Age            0.216704
sex_encoded    0.189813
Title_Mr       0.131272
Pclass         0.093829
Has_Cabin      0.044927
Title_Mrs      0.040289
Title_Miss     0.019470
Title_Rare     0.008347
dtype: float64


In [24]:
df.groupby('Title_Mr')['Survived'].mean()

Title_Mr
False    0.697861
True     0.156673
Name: Survived, dtype: float64

In [22]:
df.groupby('Title_Master')['Survived'].mean()

Title_Master
False    0.374853
True     0.575000
Name: Survived, dtype: float64

In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
preds = forest.predict(X_test)
print(confusion_matrix(y_test, preds))
print(classification_report(y_test, preds))

[[96  9]
 [19 55]]
              precision    recall  f1-score   support

           0       0.83      0.91      0.87       105
           1       0.86      0.74      0.80        74

    accuracy                           0.84       179
   macro avg       0.85      0.83      0.83       179
weighted avg       0.84      0.84      0.84       179



## Conclusion

Random forest outperformed a single decision tree, mainly by 
controlling overfitting. Feature engineering particularly 
extracting Title mattered more than adding more raw columns.